#Task 1:
Phase 1: Data Collection & Preparation
Data Sourcing: Use labeled datasets (user-reported spam, public benchmarks like Enron-Spam)
Data Balance: Ensure roughly equal spam/ham distribution to prevent bias
Data Preprocessing:
Remove headers/footers, decode HTML, lowercase conversion
Handle URLs/numbers with placeholders
Train/Validation/Test split (70/15/15%)
Phase 2: Feature Engineering with NLP
Tokenization: Split text into individual words/tokens
Text Normalization: Stemming/Lemmatization (Porter Stemmer recommended)
Feature Extraction:
TF-IDF preferred over simple Bag-of-Words
Downweights common words, amplifies important spam indicators
Phase 3: Model Training & Evaluation
Model Choice: Multinomial Naive Bayes - fast, scalable, works well with word counts
Key Techniques:
Laplace Smoothing for unknown words
Handle class imbalance with class_prior
Evaluation Metrics:
Precision: Minimize false positives (ham marked as spam)
Recall: Catch actual spam emails
F1-Score: Balanced metric
ROC-AUC: Model discrimination ability
Phase 4: Deployment & Monitoring
Deployment: Microservice pipeline (preprocessing → TF-IDF → MNB)
Monitoring: Continuous performance tracking
Retraining: Regular updates with new user feedback data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv('emails.csv')
label_col = df.columns[-1]
X = df.iloc[:, 1:-1]  
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.9545893719806763
[[704  35]
 [ 12 284]]
              precision    recall  f1-score   support

           0       0.98      0.95      0.97       739
           1       0.89      0.96      0.92       296

    accuracy                           0.95      1035
   macro avg       0.94      0.96      0.95      1035
weighted avg       0.96      0.95      0.96      1035



In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv('Reviews.csv')  

def to_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(to_sentiment)

X = df['Text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = CountVectorizer(stop_words='english')
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

tfidf = TfidfTransformer()
X_train_tfidf = tfidf.fit_transform(X_train_counts)
X_test_tfidf = tfidf.transform(X_test_counts)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
new_reviews = [
    "This product is amazing, high quality and works perfectly!",
    "It's okay, not the best but not the worst.",
    "Terrible experience, broke after first use, very disappointed."
]

new_tfidf = tfidf.transform(vectorizer.transform(new_reviews))
predictions = model.predict(new_tfidf)
print("Predicted sentiments:", predictions)


Accuracy: 0.8024118004063646
Confusion Matrix:
 [[ 2651     2 13754]
 [  160    11  8357]
 [  168    23 88565]]
Classification Report:
               precision    recall  f1-score   support

    negative       0.89      0.16      0.27     16407
     neutral       0.31      0.00      0.00      8528
    positive       0.80      1.00      0.89     88756

    accuracy                           0.80    113691
   macro avg       0.67      0.39      0.39    113691
weighted avg       0.78      0.80      0.73    113691

Predicted sentiments: ['positive' 'negative' 'negative']
